In [11]:
import numpy as np
import rice_ml
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

## Perceptron Example ##

In football, running backs (RB) and wide receivers (WR) oftentimes have overlapping roles. In some cases, a player's listed position does not always correlate with his on-field usage. This notebook seeks to use the rice_ml Perceptron Classifier class to help provide clarity in player role classification. 

In terms of prototypical roles, a RB is often tasked with taking handoffs and rushing with the ball, as well as catching short passes close to the line of scrimmage. Contrarily, a WR is prototypically tasked with catching passing further down field. With the rise of "positionless" players in the past few years, the line between the two positions has been blurried. RBs like Alvin Kamara and Christian McCaffery are catching more downfield passes, while WRs like Deebo Samuel and Cordarrele Patterson have been seen taking a lot of handoffs and lateral passes. To classify players based on the role they fill, we will use Average Depth of Target (ADOT) to see if they fill more of a WR or RB role in the passing game. 

In [22]:
# Load in NFL receiving stats dataset
csv_path = Path("..") / "data" / "NFLReceivingStats.csv"
data = pd.read_csv(csv_path)
# Display the first 5 rows of the dataset
data.head(5)

,Rank,Player Id,Player GSIS,Player College GSIS,Player Sportradar Id,Name,Team,#,POS,REC GRD,...,REC 1D,DP,FUM,FUML,CBL,CTT,CTC,YAC,YAC/REC,ADOT
0,NaN,2973,30842,NaN,9c21e9af-681c-41ef-9b00-fbc9e1668ed1,Marcedes Lewis,Broncos,89,TE,55.7,...,0,0,0,0,0,0,0,0,0.0,0.0
1,NaN,7808,39973,NaN,5c48ade7-4b9a-4757-9643-87a6e3839e2b,DeAndre Hopkins,Ravens,10,WR,82.3,...,15,1,0,0,26,19,12,68,2.8,14.5
2,NaN,7816,39983,NaN,de3421f7-2147-4835-89a5-724e87bad463,Zach Ertz,Commanders,86,TE,63.3,...,25,7,1,0,63,16,11,131,2.4,9.4
3,NaN,7844,40011,NaN,c3859e06-5f23-4302-a71b-04820a899d5f,Travis Kelce,Chiefs,87,TE,74.0,...,46,8,1,0,87,9,4,430,5.6,7.1
4,NaN,7857,40024,NaN,5f424505-f29f-433c-b3f2-1a143a04a010,Keenan Allen,Chargers,13,WR,76.1,...,51,8,2,0,96,27,12,252,3.0,8.7


As we can see when inspecting our raw data, there are several stats and positions that we do not intend to use in this example. First, to ensure ADOT data is representative of the player's usage, we limit our data to players with 10 targets. Afterwards, we filter down to only players classified as WR and RB. Then, we encode WR as our target variable for easy classification (1 if WR, -1 if RB). Finally, we split the data 80/20 for a train/test split before extracting the target and ADOT columns to use in the model. 

In [23]:
# Filter data to only include players with at least 10 targets
data = data[data["TGT"] >= 10]

# Filter data to only include WR and HB
data = data[(data["POS"] == "WR") | (data["POS"] == "HB")]

# Create a new column for the target variable (1 if WR, -1 if HB)
data["Target"] = data["POS"].apply(lambda x: 1 if x == "WR" else -1)

#count how many WR and HB are in the dataset
wr_count = data[data["POS"] == "WR"].shape[0]
hb_count = data[data["POS"] == "HB"].shape[0]
print(f"Number of WR: {wr_count}")
print(f"Number of HB: {hb_count}")

# Split data into training and testing sets
train_data = data.sample(frac=0.8, random_state=42)
test_data = data.drop(train_data.index)

# Select Target and ADOT columns for training
X = train_data[["ADOT"]].to_numpy()
y = train_data["Target"].to_numpy()

# Select Target and ADOT columns for testing
X_test = test_data[["ADOT"]].to_numpy()
y_test = test_data["Target"].to_numpy()


Number of WR: 170
Number of HB: 77


Now that our data is clean, we can fit our Perceptron Classifier Model. The available initialization arguements are eta and epochs. Eta sets the learning rate, which controls the step size during weight updates. Epochs sets the number of complete passes over the training dataset. 

The train method takes two arguements. The first is the array of feature vectors, while the second is the target labels (-1 and 1). 

In [24]:
# Fit a perceptron model to the training data
model = rice_ml.Perceptron(eta = 0.5, epochs = 100)
model.train(X, y)

After training our model, we are ready to test it on our remaining data. To do so, we simply run the test method in our class. There is only one arguement in the test method, which is the array of features. This method will give us a vector of predicted labels. 

In [30]:
# Test the model on the testing data
predictions = model.predict(X_test)

In [31]:
# Extract model bias and weights
bias = model.bias
weights = model.weights

# Print Model bias and wieghts
print(f"Model Bias: {bias:.2f}")
print(f"Model Weights: {weights}")

Model Bias: -7.00
Model Weights: [6.6]


Looking at our model summary, we see that an ADOT of 0 strongly indicates that a player is a RB. This makes sense, considering they primarily catch passes behind the line of scrimmage (LOS). A player is not classified as a WR until he achieves an ADOT of 1.06. This classification also makes sense. Many WRs catch a lot of passes laterally of behind the LOS, which brings their ADOT down, but most of their prodution still comes from downfield. Overall, both the bias and weights make sense given the context of the model. 

To measure the accuracy of our model on our test data, we run the accuracy_score function from the package. This function takes the true and predicted label vectors and calculates how often they match, yielding prediction accuracy. 

For demonstration purposes, we also use the Recall and Precision functions in the same way.

In [32]:
accuracy = rice_ml.accuracy_score(y_test, predictions)
print(f"Test Accuracy: {accuracy:.2f}")

recall = rice_ml.recall_score(y_test, predictions)
print(f"Test Recall: {recall:.2f}")

precision = rice_ml.precision_score(y_test, predictions)
print(f"Test Precision: {precision:.2f}")

#Pull inaccurate predictions
inaccurate_predictions = test_data[predictions != y_test]
print("Inaccurate Predictions:")
print(inaccurate_predictions[["Name", "POS", "ADOT", "Target"]])

Test Accuracy: 0.94
Test Recall: 1.00
Test Precision: 1.00
Inaccurate Predictions:
                    Name POS  ADOT  Target
187         Zavier Scott  HB   1.1      -1
387  Rhamondre Stevenson  HB   3.1      -1
502         Isaiah Davis  HB   1.1      -1


As we can see, the Accuracy is at 94%, while Recall and Precision are 100% for the test data (this difference is likely due to how Python treats rounding). This likely means that very few players in the test dataset are performing in roles outside their official classification. 

After extracting the players whose predicted position was inaccurate, we see that all 3 of the misclassified players are RBs. In terms of football, no WRs played a RB role in the pass game. Only 3 RBs played a WR role in the pass game. These 3 RBs are Rhamondre Stevenson, Zavier Scott, and Isaiah Davis. 

By looking further into these players' roles and responsibilities this season, we can evaluate what position they truly resemble. It is important to note that all of these players were utilized further downfield than expected. 

Zavier Scott only had 17 targets, and it appears that a few deep passes skewed his ADOT higher. His true utilization may not reflect that of a WR. 

Stevenson and Davis saw more meaningful work in the passing game. It is fair to conclude that they are, on average, utilized much further down the field than the average RB. 
